# Código 1: Detección y clasificación de outliers (sensor_error vs evento_real)

In [ ]:
# Código 1: Detección y clasificación de outliers (sensor_error vs evento_real)
# Diseñado para ejecutarse en un notebook (VSCode, Python 3.11).
# Dependencias: pandas, numpy, scikit-learn, statsmodels
# Instalar si es necesario: pip install pandas numpy scikit-learn statsmodels

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ---------------------------
# Parámetros configurables
# ---------------------------
PERIOD_H = 24                      # periodo estacional (horario → 24h)
STL_K_MAD = 4.0                    # umbral en MAD para residuo STL
JUMP_K_MAD = 6.0                   # umbral para saltos 1h basados en MAD local
ISO_CONTAMINATION = 0.01           # proporción para IsolationForest
ROLL_WINDOW_H = 24                 # ventana (horas) para calcular MAD local
PHYS_TEMP_MIN = -50.0
PHYS_TEMP_MAX = 60.0
PHYS_RSOL_MIN = 0.0
PHYS_WIND_MIN = 0.0
ANGLE_MIN = 0.0
ANGLE_MAX = 360.0
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")  # carpeta salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Lista de rutas (usa la tuya)
# ---------------------------
station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
station_paths = [os.path.expanduser(p) for p in station_paths]

# ---------------------------
# Funciones auxiliares
# ---------------------------
def robust_mad(x):
    """MAD: median absolute deviation (no-Bessel)."""
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return mad if mad > 0 else np.nanmean(np.abs(x - med))  # fallback

def ensure_datetime_index(df, datetime_col=None, freq='h'):
    """Asegura índice datetime con frecuencia horaria y reindexa dejando NaNs donde faltan timestamps."""
    if datetime_col is None:
        # intentar encontrar columna timestamp
        candidates = [c for c in df.columns if c.lower() in ("datetime","fecha","time","timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Pasa el nombre con datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    # rellenar para rango completo
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df


def physical_checks(df):
    """Marca errores físicos obvios."""
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['reason'] = ""
    # R.Sol. < 0
    if 'R.Sol.' in df.columns:
        mask = (df['R.Sol.'] < PHYS_RSOL_MIN) & (~df['R.Sol.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "R.Sol. < 0; "
    # Veloc < 0
    if 'Veloc.' in df.columns:
        mask = (df['Veloc.'] < PHYS_WIND_MIN) & (~df['Veloc.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Veloc < 0; "
    # Angulo/Direc fuera de rango
    for col in ['Direc.','Angulo(en grados)','Angle','Dirección']:
        if col in df.columns:
            mask = (~df[col].isna()) & ((df[col] < ANGLE_MIN) | (df[col] >= ANGLE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{col} fuera [0,360); "
    # Temp extrema
    if 'Temp' in df.columns:
        mask = (~df['Temp'].isna()) & ((df['Temp'] < PHYS_TEMP_MIN) | (df['Temp'] > PHYS_TEMP_MAX))
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Temp extrema; "
    # O3 negativa
    for o3_col in ['O3','O₃','Ozone']:
        if o3_col in df.columns:
            mask = (~df[o3_col].isna()) & (df[o3_col] < 0)
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{o3_col} < 0; "
    return flags

def stl_outliers(df, col='O3', period=PERIOD_H, k=STL_K_MAD):
    """Detecta outliers en la serie usando STL residuo + MAD"""
    res = pd.Series(False, index=df.index)
    try:
        s = df[col].dropna()
        if len(s) > period*2:
            stl = STL(s, period=period, robust=True)
            out = stl.fit()
            resid = out.resid
            mad_resid = robust_mad(resid)
            threshold = k * mad_resid
            mask = np.abs(resid) > threshold
            res.loc[mask.index] = mask.values
    except Exception as e:
        # si falla, devolvemos serie False
        print(f"STL failed for {col}: {e}")
    return res

def rolling_jump_outliers(df, col='O3', window=ROLL_WINDOW_H, k=JUMP_K_MAD):
    """Detecta saltos bruscos 1h comparando diffs y MAD local."""
    res = pd.Series(False, index=df.index)
    s = df[col]
    # diff 1h (forward/backward)
    diff = s.diff().abs()
    # calc rolling MAD of diffs
    roll_mad = diff.rolling(window=window, min_periods=1, center=True).apply(lambda x: robust_mad(x), raw=True)
    thr = k * roll_mad
    mask = (diff > thr) & (~diff.isna())
    res.loc[mask.index] = mask.values
    return res

def isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION):
    """IsolationForest multivariado (retorna boolean series)."""
    if features is None:
        candidates = ['O3','NO','NO2','NOx','Temp','Veloc.','R.Sol.']
        features = [c for c in candidates if c in df.columns]
    res = pd.Series(False, index=df.index)
    sub = df[features].copy()
    valid_idx = sub.dropna(how='all').index
    if len(valid_idx) < 20:
        return res
    sub = sub.loc[valid_idx]
    try:
        # Rellenamos hacia adelante, hacia atrás y finalmente ceros si aún quedan NaN
        sub = sub.ffill().bfill().fillna(0)
        iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
        pred = iso.fit_predict(sub.values)
        out_idx = sub.index[pred == -1]
        res.loc[out_idx] = True
    except Exception as e:
        print("IsolationForest failed:", e)
    return res

def neighbor_support(df, idx, col='O3', window=3, rel_tol=0.5):
    """
    Evalúa si un pico en idx tiene soporte en vecinos (vecinos muestran cambios coherentes).
    window: número total de puntos a mirar (debe ser impar), default 3 -> t-1, t, t+1
    rel_tol: fracción del tamaño del cambio que consideraremos 'soporte'
    """
    half = (window - 1) // 2
    i_pos = df.index.get_loc(idx)
    start = max(0, i_pos - half)
    end = min(len(df) - 1, i_pos + half)
    vals = df[col].iloc[start:end+1].values
    center = df[col].iloc[i_pos]
    # si la mayoría de vecinos están NaN → no hay soporte
    if np.isnan(vals).sum() >= half + 1:
        return False
    # support heuristic: si cambio del centro respecto a median vecino es mayor que
    # median(neighbors) * rel_tol -> we consider neighbors support if they change similarly
    med_neighbors = np.nanmedian(np.concatenate([vals[:half], vals[half+1:]])) if len(vals) > 1 else np.nan
    if np.isnan(med_neighbors):
        return False
    # if both neighbors changed in same direction as center relative to older values -> support
    # Simpler: if abs(center - med_neighbors) < abs(center) * rel_tol => we say has support
    if abs(center - med_neighbors) <= abs(center) * rel_tol:
        return True
    return False

# ---------------------------
# Pipeline para una estación
# ---------------------------
def process_station(path, save_out=True, verbose=True):
    """Carga CSV de estación, detecta outliers y clasifica sensor_error/evento_real/doubtful.
       Devuelve dataframe con columnas nuevas y guarda CSV con sufijo _outliers.csv"""
    station_name = Path(path).stem
    if verbose:
        print("Procesando:", station_name)
    df_raw = pd.read_csv(path)
    df = ensure_datetime_index(df_raw, datetime_col=None, freq='h')
    # Inicializar flags
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""
    # physical checks
    phys = physical_checks(df)
    flags['physical_error'] = phys['physical_error']
    flags['physical_reason'] = phys['reason']

    # detectar outliers univariados por STL (O3 si existe)
    o3_col = next((c for c in df.columns if c in ['O3']), None)
    if o3_col is None:
        print(f"Warning: {station_name} no tiene columna O3 reconocida. Saltando STL/jump para O3.")
        stl_mask = pd.Series(False, index=df.index)
        jump_mask = pd.Series(False, index=df.index)
    else:
        stl_mask = stl_outliers(df, col=o3_col, period=PERIOD_H, k=STL_K_MAD)
        jump_mask = rolling_jump_outliers(df, col=o3_col, window=ROLL_WINDOW_H, k=JUMP_K_MAD)

    flags['outlier_stl'] = stl_mask
    flags['outlier_jump'] = jump_mask

    # Isolation Forest multivariado
    iso_mask = isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION)
    flags['outlier_iso'] = iso_mask

    # Clasificación heurística sensor_error vs evento_real vs doubtful
    flags['sensor_error'] = False
    flags['evento_real'] = False
    flags['doubtful'] = False

    # reglas básicas
    for ts in df.index:
        reasons = []
        # physical error overrides
        if flags.at[ts, 'physical_error']:
            flags.at[ts, 'sensor_error'] = True
            reasons.append("physical")
            continue
        # detector combinados
        is_stl = bool(flags.at[ts, 'outlier_stl'])
        is_jump = bool(flags.at[ts, 'outlier_jump'])
        is_iso = bool(flags.at[ts, 'outlier_iso'])
        any_det = is_stl or is_jump or is_iso
        if not any_det:
            # no outlier -> nothing to do
            continue
        # if multiple detectors agree → more fiable
        n_det = sum([is_stl, is_jump, is_iso])
        # check neighbour support only for O3
        support = False
        if o3_col is not None and not np.isnan(df.at[ts,o3_col]):
            try:
                support = neighbor_support(df, ts, col=o3_col, window=3, rel_tol=0.6)
            except Exception:
                support = False
        # Heurística:
        # - Si hay detección pero sin soporte en vecinos y detección es 'aislada' -> probable sensor_error
        # - Si varios detectores y hay soporte / hay picos también en NO/NO2/NOx/Temp/R.Sol -> evento_real
        related_vars = ['NO','NO2','NOx','Temp','R.Sol.','Veloc.']
        related_present = any([(v in df.columns) and (not pd.isna(df.at[ts,v])) for v in related_vars])
        # check if related vars also spike (simple: compare to rolling median)
        related_spike = False
        for v in related_vars:
            if v in df.columns:
                val = df.at[ts,v]
                if pd.isna(val):
                    continue
                rolling_med = df[v].rolling(window=ROLL_WINDOW_H, min_periods=1, center=True).median().iloc[df.index.get_loc(ts)]
                if not pd.isna(rolling_med):
                    if abs(val - rolling_med) > 2 * robust_mad(df[v].dropna()):  # simple threshold
                        related_spike = True
                        break
        # classification rules:
        if n_det >= 2 and related_spike:
            flags.at[ts, 'evento_real'] = True
            reasons.append("multi_det + related_spike")
        else:
            # if isolated detection and no neighbor support => sensor error
            if not support and n_det >= 1:
                flags.at[ts, 'sensor_error'] = True
                reasons.append("isolated_no_support")
            else:
                # ambiguous: iso true but no clear support or multiple detectors but no related spike
                flags.at[ts, 'doubtful'] = True
                reasons.append("ambiguous")
        # end loop

    # Crear columna de imputation: si sensor_error True -> poner NaN (preparar imputación)
    df_proc = df.copy()
    if o3_col is not None:
        df_proc['O3_for_impute'] = df_proc[o3_col].where(~flags['sensor_error'], np.nan)

    # adjuntar flags al df final
    out_df = pd.concat([df_proc, flags], axis=1)

    # guardar CSV con sufijo
    if save_out:
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_outliers.csv")
        out_df.to_csv(out_path, index=True)
        if verbose:
            print(f"Guardado: {out_path}")
    return out_df

# ---------------------------
# Procesar todas las estaciones y compilar doubtfuls
# ---------------------------
all_doubtful = []
all_summaries = []
for p in station_paths:
    try:
        df_out = process_station(p, save_out=True, verbose=True)
        # extraer filas dudosas
        if 'doubtful' in df_out.columns:
            doubtful_rows = df_out[df_out['doubtful'] == True].copy()
            if not doubtful_rows.empty:
                # añadir columna estación
                doubtful_rows['station_file'] = Path(p).stem
                all_doubtful.append(doubtful_rows)
        # summary counts
        summary = {
            'station': Path(p).stem,
            'n_rows': len(df_out),
            'n_physical_error': int(df_out['physical_error'].sum()) if 'physical_error' in df_out.columns else 0,
            'n_outlier_stl': int(df_out['outlier_stl'].sum()) if 'outlier_stl' in df_out.columns else 0,
            'n_outlier_jump': int(df_out['outlier_jump'].sum()) if 'outlier_jump' in df_out.columns else 0,
            'n_outlier_iso': int(df_out['outlier_iso'].sum()) if 'outlier_iso' in df_out.columns else 0,
            'n_sensor_error': int(df_out['sensor_error'].sum()) if 'sensor_error' in df_out.columns else 0,
            'n_evento_real': int(df_out['evento_real'].sum()) if 'evento_real' in df_out.columns else 0,
            'n_doubtful': int(df_out['doubtful'].sum()) if 'doubtful' in df_out.columns else 0,
        }
        all_summaries.append(summary)
    except Exception as e:
        print(f"Error procesando {p}: {e}")

# guardar doubtfuls combinados
if all_doubtful:
    combined = pd.concat(all_doubtful, axis=0)
    combined_path = os.path.join(OUTPUT_DIR, "doubtful_cases_combined.csv")
    combined.to_csv(combined_path, index=True)
    print(f"CSV combinado de casos dudosos guardado en: {combined_path}")
# guardar resumen
summary_df = pd.DataFrame(all_summaries)
summary_path = os.path.join(OUTPUT_DIR, "outliers_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

print("Proceso completado. Revisa la carpeta de salida para CSVs por estación, summary y casos dudosos.")

Procesando: T1_E1_Alicante


# Código 2 (Versión A — IterativeImputer multivariante)

In [ ]:
# Código 2 (Versión A — IterativeImputer multivariante)
# Dep: pandas, numpy, scikit-learn
# pip install pandas numpy scikit-learn

import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ---------- PARÁMETROS (ajustables) ----------
SHORT_GAP_H = 6         # <= -> interpolación temporal
MID_GAP_H = 72          # <= -> IterativeImputer; > -> mediana estacional
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_imputed_iterative")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Rutas de estaciones (usa tus rutas)
station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
station_paths = [os.path.expanduser(p) for p in station_paths]

O3_CANDIDATES = ['O3','O₃','Ozone']
ANGLE_COLS = ['Direc.','Angulo(en grados)','Dirección','Angle']
CONTINUOUS_CANDIDATES = ['NO','NO2','NOx','Temp','R.Sol.','Veloc.','Dist.']

# ----------------- UTILIDADES -----------------
def ensure_datetime_index(df, datetime_col=None, freq='H'):
    if datetime_col is None:
        candidates = [c for c in df.columns if c.lower() in ("datetime","fecha","time","timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Pasa el nombre con datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq=freq)
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df

def nan_group_lengths(series):
    isna = series.isna().astype(int).values
    n = len(isna)
    group_len = np.zeros(n, dtype=int)
    i = 0
    while i < n:
        if isna[i] == 1:
            j = i
            while j < n and isna[j] == 1:
                j += 1
            length = j - i
            group_len[i:j] = length
            i = j
        else:
            i += 1
    return group_len

def add_cyclical_time_features(df):
    idx = df.index
    hour = idx.hour
    month = idx.month
    week = idx.isocalendar().week.astype(int)
    dayofyear = idx.dayofyear
    df['hour_sin'] = np.sin(2*np.pi*hour/24)
    df['hour_cos'] = np.cos(2*np.pi*hour/24)
    df['month_sin'] = np.sin(2*np.pi*(month-1)/12)
    df['month_cos'] = np.cos(2*np.pi*(month-1)/12)
    df['week_sin'] = np.sin(2*np.pi*(week-1)/52)
    df['week_cos'] = np.cos(2*np.pi*(week-1)/52)
    df['doy_sin'] = np.sin(2*np.pi*(dayofyear-1)/365.25)
    df['doy_cos'] = np.cos(2*np.pi*(dayofyear-1)/365.25)
    return df

def seasonal_median_fill(df, var):
    grp = df.groupby([df.index.month, df.index.hour])[var].transform('median')
    filled = df[var].copy()
    mask = filled.isna()
    filled.loc[mask] = grp.loc[mask]
    filled = filled.fillna(df[var].median())
    return filled

# ----------------- Imputer factory -----------------
def make_iterative_imputer(estimator_name='rf', n_estimators=100, max_iter=10, random_state=RANDOM_STATE, tol=1e-3):
    """Crea IterativeImputer con el estimador deseado."""
    if estimator_name == 'rf':
        base_est = RandomForestRegressor(n_estimators=n_estimators, n_jobs=-1, random_state=random_state)
    elif estimator_name == 'bayes':
        base_est = BayesianRidge()
    else:
        raise ValueError("estimator_name must be 'rf' or 'bayes'")
    imputer = IterativeImputer(estimator=base_est, max_iter=max_iter, random_state=random_state, tol=tol, initial_strategy='mean', sample_posterior=False)
    return imputer

# ----------------- Imputación por estación con IterativeImputer -----------------
def impute_station_iterative(df, station_name=None, short_gap_h=SHORT_GAP_H, mid_gap_h=MID_GAP_H,
                             estimator_name='rf', n_estimators=100, max_iter=10, log_imputations=False):
    """
    Imputa DataFrame de una estación con:
     - short gaps: time interpolation
     - mid gaps: IterativeImputer (multivariante)
     - long gaps: seasonal median (month,hour)
    Devuelve df_imputed y opcionalmente guarda CSV de log de imputaciones.
    """
    df = df.copy()
    # detectar O3 col a usar
    o3_col = next((c for c in df.columns if c in O3_CANDIDATES), None)
    target_o3_col = 'O3_for_impute' if 'O3_for_impute' in df.columns else o3_col

    conts = [c for c in CONTINUOUS_CANDIDATES if c in df.columns]
    if target_o3_col and target_o3_col not in conts:
        conts = [target_o3_col] + conts

    angle_present = [c for c in ANGLE_COLS if c in df.columns]

    # --- CREAR COLUMNAS DERIVADAS (sin/cos) ANTES DE CALCULAR GRUPOS NaN ---
    for ang_col in angle_present:
        sin_col = f"{ang_col}_sin"
        cos_col = f"{ang_col}_cos"
        df[sin_col] = np.sin(np.deg2rad(df[ang_col]))
        df[cos_col] = np.cos(np.deg2rad(df[ang_col]))
        # añadir a conts para que sean imputadas también
        if sin_col not in conts:
            conts += [sin_col, cos_col]

    # añadir features cíclicas
    df = add_cyclical_time_features(df)

    # --- AHORA CALCULAR GRUPOS NaN ORIGINALES (incluye todas las columnas en conts) ---
    original_nan_groups = {col: nan_group_lengths(df[col]) if col in df.columns else np.zeros(len(df),dtype=int)
                           for col in conts}

    # inicializar indicadores
    for col in conts + angle_present:
        df[f"imputed_flag_{col}"] = False
        df[f"imputed_method_{col}"] = 'original'

    # registro de imputaciones (opcional)
    imput_log_rows = []

    # 1) Interpolación corta
    for col in conts:
        if col not in df.columns:
            continue
        ng = original_nan_groups[col]
        mask_short = (ng > 0) & (ng <= short_gap_h)
        if mask_short.any():
            interp_full = df[col].interpolate(method='time', limit_direction='both')
            idxs = df.index[mask_short]
            df.loc[idxs, col] = interp_full.loc[idxs]
            df.loc[idxs, f"imputed_flag_{col}"] = True
            df.loc[idxs, f"imputed_method_{col}"] = 'interp_short'
            if log_imputations:
                for ts in idxs:
                    imput_log_rows.append({'station': station_name or 'unknown', 'datetime': ts, 'column': col,
                                           'method':'interp_short','imputed_value': df.at[ts,col]})

    # 2) IterativeImputer for mid gaps (aplicado a todas las columnas numéricas simultáneamente)
    cyc_cols = [c for c in df.columns if c.startswith(('hour_','month_','week_','doy_'))]
    num_cols = list(dict.fromkeys(conts + cyc_cols))  # unique preserving order
    # preparar array para imputar (copiar)
    impute_input = df[num_cols].copy()

    # crear el imputer
    imputer = make_iterative_imputer(estimator_name=estimator_name, n_estimators=n_estimators, max_iter=max_iter)

    # Fit-transform (avisa si dataset demasiado pequeño)
    n_non_nan_rows = impute_input.dropna(how='all').shape[0]
    if n_non_nan_rows < 10:
        print(f"[{station_name}] Pocos datos observados ({n_non_nan_rows}) para IterativeImputer — se saltará.")
    else:
        try:
            imputed_array = imputer.fit_transform(impute_input)
            imputed_df = pd.DataFrame(imputed_array, index=impute_input.index, columns=num_cols)
            # marcar cuales valores fueron imputados por el iterativo (comparamos con orig NaN y que original gap <= mid_gap)
            for col in conts:
                if col not in imputed_df.columns:
                    continue
                orig_nan = (original_nan_groups[col] > 0)
                # mask for mid gaps only
                mid_mask = (original_nan_groups[col] > short_gap_h) & (original_nan_groups[col] <= mid_gap_h)
                # entries where original was NaN and mid_mask true -> these should be considered filled by iterative imputer
                target_idxs = df.index[mid_mask]
                if len(target_idxs) > 0:
                    for ts in target_idxs:
                        if pd.isna(df.at[ts, col]):   # aún NaN (no se sobreescribió en paso anterior)
                            new_val = imputed_df.at[ts, col]
                            df.at[ts, col] = new_val
                            df.at[ts, f"imputed_flag_{col}"] = True
                            df.at[ts, f"imputed_method_{col}"] = f'iterative_{estimator_name}'
                            if log_imputations:
                                imput_log_rows.append({'station': station_name or 'unknown', 'datetime': ts, 'column': col,
                                                       'method':f'iterative_{estimator_name}','imputed_value': new_val})
        except Exception as e:
            print(f"[{station_name}] IterativeImputer falló: {e}")

    # 3) Gaps largos -> mediana estacional por station
    for col in conts:
        if col not in df.columns:
            continue
        ng = original_nan_groups[col]
        long_mask = (ng > mid_gap_h)
        if long_mask.any():
            long_idxs = df.index[long_mask]
            filled = seasonal_median_fill(df, col)
            for ts in long_idxs:
                if pd.isna(df.at[ts, col]):
                    df.at[ts, col] = filled.at[ts]
                    df.at[ts, f"imputed_flag_{col}"] = True
                    df.at[ts, f"imputed_method_{col}"] = 'seasonal_median'
                    if log_imputations:
                        imput_log_rows.append({'station': station_name or 'unknown', 'datetime': ts, 'column': col,
                                               'method':'seasonal_median','imputed_value': df.at[ts, col]})

    # 4) reconstrucción angulos desde sin/cos
    for ang_col in angle_present:
        sin_col = f"{ang_col}_sin"
        cos_col = f"{ang_col}_cos"
        if sin_col in df.columns and cos_col in df.columns:
            ang_recon = np.rad2deg(np.arctan2(df[sin_col].values, df[cos_col].values)) % 360
            df[f"{ang_col}_recon"] = ang_recon
            orig_nan_mask = df[ang_col].isna() & (~pd.isna(df[f"{ang_col}_recon"]))
            for ts in df.index[orig_nan_mask]:
                df.at[ts, ang_col] = df.at[ts, f"{ang_col}_recon"]
                df.at[ts, f"imputed_flag_{ang_col}"] = True
                df.at[ts, f"imputed_method_{ang_col}"] = 'reconstructed_from_components'
                if log_imputations:
                    imput_log_rows.append({'station': station_name or 'unknown', 'datetime': ts, 'column': ang_col,
                                           'method':'reconstructed_from_components','imputed_value': df.at[ts, ang_col]})

    # 5) Categoricals: forward/backfill (corregido: sin method)
    for cat in ['Estacion','Transecto']:
        if cat in df.columns:
            df[cat] = df[cat].ffill().bfill()
            if df[cat].isna().any():
                mode = df[cat].mode(dropna=True)
                if not mode.empty:
                    df[cat] = df[cat].fillna(mode.iloc[0])

    # 6) Guardar log si requerido
    if log_imputations and imput_log_rows:
        log_df = pd.DataFrame(imput_log_rows)
        out_log = os.path.join(OUTPUT_DIR, f"{station_name}_imputation_log_iterative.csv")
        log_df.to_csv(out_log, index=False)
        print(f"Log de imputaciones guardado: {out_log}")

    return df

def impute_by_station_iterative(station_paths_list, short_gap_h=SHORT_GAP_H, mid_gap_h=MID_GAP_H,
                                estimator_name='rf', n_estimators=100, max_iter=10, log_imputations=False):
    summaries = []
    for p in station_paths_list:
        pth = Path(p)
        station_name = pth.stem
        # preferir archivo _outliers.csv si existe (generado por Código1)
        out_candidate = pth.parent / f"{pth.stem}_outliers.csv"
        if out_candidate.exists():
            df = pd.read_csv(out_candidate, index_col=0, parse_dates=True)
        else:
            df = pd.read_csv(pth, parse_dates=True)
        try:
            df = ensure_datetime_index(df)
        except Exception:
            df.index.name = 'datetime'
        df_imputed = impute_station_iterative(df, station_name=station_name, short_gap_h=short_gap_h,
                                              mid_gap_h=mid_gap_h, estimator_name=estimator_name,
                                              n_estimators=n_estimators, max_iter=max_iter,
                                              log_imputations=log_imputations)
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_imputed_iterative.csv")
        df_imputed.to_csv(out_path, index=True)
        print(f"Guardado (iterative, estación): {out_path}")
        summaries.append({'station':station_name, 'n_rows': len(df_imputed)})
    pd.DataFrame(summaries).to_csv(os.path.join(OUTPUT_DIR, "impute_by_station_iterative_summary.csv"), index=False)
    print("Resumen imputación por estación (iterative) guardado.")
    return summaries

# ----------------- Imputación GLOBAL (IterativeImputer sobre todas las estaciones) -----------------
def impute_global_iterative(station_paths_list, short_gap_h=SHORT_GAP_H, mid_gap_h=MID_GAP_H,
                            estimator_name='rf', n_estimators=100, max_iter=10, log_imputations=False):
    # cargar todos
    dfs = []
    for p in station_paths_list:
        pth = Path(p)
        station_name = pth.stem
        out_candidate = pth.parent / f"{pth.stem}_outliers.csv"
        if out_candidate.exists():
            df = pd.read_csv(out_candidate, index_col=0, parse_dates=True)
        else:
            df = pd.read_csv(pth, parse_dates=True)
        try:
            df = ensure_datetime_index(df)
        except Exception:
            df.index.name = 'datetime'
        df['__station_name'] = station_name
        dfs.append(df)
    combined = pd.concat(dfs, axis=0)
    # asegurarse de columna station_name
    if '__station_name' in combined.columns:
        combined = combined.rename(columns={'__station_name':'station_name'}) if 'station_name' not in combined.columns else combined

    # encode station
    le = LabelEncoder()
    combined['station_code'] = le.fit_transform(combined['station_name'].astype(str))

    # preparar conts y angles
    o3_col = next((c for c in combined.columns if c in O3_CANDIDATES), None)
    target_o3_col = 'O3_for_impute' if 'O3_for_impute' in combined.columns else o3_col
    conts = [c for c in CONTINUOUS_CANDIDATES if c in combined.columns]
    if target_o3_col and target_o3_col not in conts:
        conts = [target_o3_col] + conts
    angle_present = [c for c in ANGLE_COLS if c in combined.columns]

    # --- CREAR COLUMNAS DERIVADAS (sin/cos) ANTES DE CALCULAR GRUPOS NaN ---
    for ang_col in angle_present:
        sin_col = f"{ang_col}_sin"
        cos_col = f"{ang_col}_cos"
        combined[sin_col] = np.sin(np.deg2rad(combined[ang_col]))
        combined[cos_col] = np.cos(np.deg2rad(combined[ang_col]))
        if sin_col not in conts:
            conts += [sin_col, cos_col]

    combined = add_cyclical_time_features(combined)

    # --- AHORA CALCULAR GRUPOS NaN ORIGINALES por estación (incluye todas las columnas en conts) ---
    original_nan_groups = {}
    for station in combined['station_name'].unique():
        g = combined[combined['station_name'] == station]
        original_nan_groups[station] = {
            col: nan_group_lengths(g[col]) if col in g.columns else np.zeros(len(g), dtype=int)
            for col in conts
        }

    # indicadores
    for col in conts + angle_present:
        combined[f"imputed_flag_{col}"] = False
        combined[f"imputed_method_{col}"] = 'original'

    imput_log_rows = []

    # 1) Interpolación corta por estación
    for station in combined['station_name'].unique():
        mask_station = combined['station_name'] == station
        g = combined.loc[mask_station]
        for col in conts:
            if col not in g.columns:
                continue
            ng = original_nan_groups[station][col]
            mask_short = (ng > 0) & (ng <= short_gap_h)
            if mask_short.any():
                interp_full = g[col].interpolate(method='time', limit_direction='both')
                idxs = g.index[mask_short]
                combined.loc[idxs, col] = interp_full.loc[idxs]
                combined.loc[idxs, f"imputed_flag_{col}"] = True
                combined.loc[idxs, f"imputed_method_{col}"] = 'interp_short'
                if log_imputations:
                    for ts in idxs:
                        imput_log_rows.append({'station': station, 'datetime': ts, 'column': col,
                                               'method':'interp_short', 'imputed_value': combined.at[ts, col]})

    # 2) IterativeImputer global para gaps medios
    cyc_cols = [c for c in combined.columns if c.startswith(('hour_','month_','week_','doy_'))]
    num_cols = list(dict.fromkeys(conts + cyc_cols + ['station_code']))
    impute_input = combined[num_cols].copy()

    n_non_nan = impute_input.dropna(how='all').shape[0]
    if n_non_nan < 20:
        print("Pocos datos no-nulos para IterativeImputer global — se saltará.")
    else:
        imputer = make_iterative_imputer(estimator_name=estimator_name, n_estimators=n_estimators, max_iter=max_iter)
        try:
            imputed_array = imputer.fit_transform(impute_input)
            imputed_df = pd.DataFrame(imputed_array, index=impute_input.index, columns=num_cols)
            # asignar sólo a posiciones que correspondían a mid gaps (por estación)
            for station in combined['station_name'].unique():
                mask_station = combined['station_name'] == station
                g_idx = combined.index[mask_station]
                for col in conts:
                    if col not in imputed_df.columns:
                        continue
                    ng = original_nan_groups[station][col]
                    # mid positions relative to the station group: need station-local boolean mask aligned to global index
                    local_mid_mask = (ng > short_gap_h) & (ng <= mid_gap_h)
                    # map these local positions to global index
                    local_indices = g_idx[local_mid_mask]
                    for ts in local_indices:
                        if pd.isna(combined.at[ts, col]):
                            combined.at[ts, col] = imputed_df.at[ts, col]
                            combined.at[ts, f"imputed_flag_{col}"] = True
                            combined.at[ts, f"imputed_method_{col}"] = f'iterative_{estimator_name}_global'
                            if log_imputations:
                                imput_log_rows.append({'station': station, 'datetime': ts, 'column': col,
                                                       'method': f'iterative_{estimator_name}_global',
                                                       'imputed_value': combined.at[ts, col]})
        except Exception as e:
            print("IterativeImputer global falló:", e)

    # 3) Gaps largos -> mediana estacional por station
    for station in combined['station_name'].unique():
        mask_station = combined['station_name'] == station
        g = combined.loc[mask_station]
        for col in conts:
            ng = original_nan_groups[station][col]
            long_mask = (ng > mid_gap_h)
            if long_mask.any():
                local_idx = g.index[long_mask]
                filled = seasonal_median_fill(g, col)
                for ts in local_idx:
                    if pd.isna(combined.at[ts, col]):
                        combined.at[ts, col] = filled.at[ts]
                        combined.at[ts, f"imputed_flag_{col}"] = True
                        combined.at[ts, f"imputed_method_{col}"] = 'seasonal_median_global'
                        if log_imputations:
                            imput_log_rows.append({'station': station, 'datetime': ts, 'column': col,
                                                   'method': 'seasonal_median_global',
                                                   'imputed_value': combined.at[ts, col]})

    # reconstrucción angulos
    for ang_col in angle_present:
        sin_col = f"{ang_col}_sin"
        cos_col = f"{ang_col}_cos"
        ang_recon = np.rad2deg(np.arctan2(combined[sin_col].values, combined[cos_col].values)) % 360
        combined[f"{ang_col}_recon"] = ang_recon
        orig_nan_mask = combined[ang_col].isna() & (~pd.isna(combined[f"{ang_col}_recon"]))
        for ts in combined.index[orig_nan_mask]:
            combined.at[ts, ang_col] = combined.at[ts, f"{ang_col}_recon"]
            combined.at[ts, f"imputed_flag_{ang_col}"] = True
            combined.at[ts, f"imputed_method_{ang_col}"] = 'reconstructed_from_components'
            if log_imputations:
                imput_log_rows.append({'station': combined.at[ts, 'station_name'], 'datetime': ts, 'column': ang_col,
                                       'method': 'reconstructed_from_components',
                                       'imputed_value': combined.at[ts, ang_col]})

    # guardar resultado global
    out_path = os.path.join(OUTPUT_DIR, "ALL_stations_imputed_iterative_global.csv")
    combined.to_csv(out_path, index=True)
    print("Guardado (iterative global):", out_path)

    # guardar log si existe
    if log_imputations and imput_log_rows:
        log_df = pd.DataFrame(imput_log_rows)
        out_log = os.path.join(OUTPUT_DIR, "imputation_log_iterative_global.csv")
        log_df.to_csv(out_log, index=False)
        print("Log de imputaciones global guardado:", out_log)

    return combined

# ----------------- Uso (descomentar para ejecutar) -----------------




# Ejecutar imputación por estación
# summaries = impute_by_station_iterative(station_paths, short_gap_h=SHORT_GAP_H, mid_gap_h=MID_GAP_H,
#                                        estimator_name='rf', n_estimators=100, max_iter=10, log_imputations=True)





# Ejecutar imputación global (todas las estaciones juntas)
# combined_global = impute_global_iterative(station_paths, short_gap_h=SHORT_GAP_H, mid_gap_h=MID_GAP_H,
#                                          estimator_name='rf', n_estimators=100, max_iter=10, log_imputations=True)






# Nota final:
# - Si tienes muchos datos y te interesa velocidad/memoria, prueba estimator_name='bayes' (BayesianRidge) o reducir n_estimators.
# - Ajusta max_iter según convergencia y tiempo.
# - El código respeta la regla de no eliminar filas; las imputaciones quedan reflejadas en columnas imputed_flag_*/imputed_method_*.